# Notebook 03 - Yelp SVD Baseline Model

This notebook implements a matrix factorization SVD baseline model using the `surprise` library. The MF SVD model works by training on the temporal train split, fine-tuning on validation splits, and evaluating on the test splits using our ranking metrics.

In [1]:
!pip install "numpy<2" scikit-surprise

In [2]:
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive')
origin_dir = '/content/drive/MyDrive/rec_system'

train_df = pd.read_parquet(f"{origin_dir}/train_reviews.parquet")
val_df   = pd.read_parquet(f"{origin_dir}/val_reviews.parquet")
test_df  = pd.read_parquet(f"{origin_dir}/test_reviews.parquet")

print(f"Train: {len(train_df)} reviews")
print(f"Val: {len(val_df)} reviews")
print(f"Test: {len(test_df)} reviews")

Mounted at /content/drive
Train: 1918257 reviews
Val: 153250 reviews
Test: 153746 reviews


In [3]:
# pre-SVD training...
# rating range given to reader to normalize properly
# Dataset load_from_df to convert train df to surprise dataset format
# and finally, build trainset object for fitting

reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(train_df[["user_id", "business_id", "stars"]], reader)
trainset = train_data.build_full_trainset()
print(f"Trainset: {trainset.n_users} users, {trainset.n_items} businesses")

Trainset: 157358 users, 39108 businesses


In [4]:
# create SVD model with hyperparameters

svd = SVD(n_factors=128, # size of latent space (so 128-dim vectors match with user + item vectors)
          n_epochs=20,
          lr_all=0.005,
          reg_all=0.02,
          random_state=42,
          verbose=True)

svd.fit(trainset)
print("Training completed.")

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Training completed.


In [5]:
def compute_rmse(model, df):
    preds = [
        model.predict(row["user_id"], row["business_id"]).est
        for _, row in df.iterrows()
    ]
    actuals = df["stars"].values
    rmse = np.sqrt(np.mean((np.array(preds) - actuals) ** 2))
    return rmse

val_rmse  = compute_rmse(svd, val_df)
print(f"Val RMSE:  {val_rmse:.4f}")

Val RMSE:  1.2282


In [ ]:
all_business_ids = set(train_df["business_id"].unique())

def evaluate_sampled(model, val_df, train_df, n_negatives=99, ks=[5, 10, 20]):
    user_train_items = train_df.groupby("user_id")["business_id"].apply(set).to_dict()
    all_businesses = list(all_business_ids)

    results = {k: {"precision": [], "recall": [], "ndcg": []} for k in ks}

    for _, row in val_df.iterrows():
        user_id = row["user_id"]
        pos_item = row["business_id"]
        label = row["label"]

        if label != 1.0:
            continue

        seen = user_train_items.get(user_id, set())
        negatives = [b for b in np.random.choice(all_businesses, n_negatives * 2, replace=False)
                     if b not in seen and b != pos_item][:n_negatives]

        candidates = [(pos_item, model.predict(user_id, pos_item).est, 1.0)] + \
                     [(neg, model.predict(user_id, neg).est, 0.0) for neg in negatives]

        candidates.sort(key=lambda x: x[1], reverse=True)

        for k in ks:
            top_k = candidates[:k]
            hit = any(item == pos_item for item, _, _ in top_k)

            results[k]["precision"].append(1.0 / k if hit else 0)

            results[k]["recall"].append(1.0 if hit else 0)

            for rank, (item, _, _) in enumerate(top_k):
                if item == pos_item:
                    results[k]["ndcg"].append(1.0 / np.log2(rank + 2))
                    break
            else:
                results[k]["ndcg"].append(0.0)

    for k in ks:
        p = np.mean(results[k]["precision"])
        r = np.mean(results[k]["recall"])
        n = np.mean(results[k]["ndcg"])
        print(f"k = {k} | Precision: {p:.4f} | Recall: {r:.4f} | NDCG: {n:.4f}")

evaluate_sampled(svd, val_df, train_df)


In [ ]:
from google.colab import files
files.upload()
yelp_open_restaurants_df = pd.read_csv("yelp_open.csv")

def test_svd_recommendations(model, user_id, businesses_df, train_df, k=10):
    # Get businesses the user has already visited
    seen = set(train_df[train_df["user_id"] == user_id]["business_id"])

    # Score all unseen businesses
    candidates = [
        (row["business_id"], row["name"], row["city"], row["state"],
         model.predict(user_id, row["business_id"]).est)
        for _, row in businesses_df.iterrows()
        if row["business_id"] not in seen
    ]

    # Sort by predicted score and take top-k
    top_k = sorted(candidates, key=lambda x: x[4], reverse=True)[:k]

    print(f"Top {k} recommendations for user {user_id}:\n")
    for i, (bid, name, city, state, score) in enumerate(top_k, 1):
        print(f"{i}. {name} ({city}, {state}) — predicted score: {score:.2f}")

# Pick a random user from val set and test
sample_user = val_df["user_id"].iloc[0]
test_svd_recommendations(svd, sample_user, yelp_open_restaurants_df, train_df, k=10)

In [ ]:
sample_user = val_df["user_id"].iloc[82199]
user_info = train_df[train_df["user_id"] == sample_user]
print(f"User ID: {sample_user}")
print(f"Total reviews in train: {len(user_info)}")
print(f"Average stars given: {user_info['stars'].mean():.2f}")
print(f"Cities visited: {yelp_open_restaurants_df[yelp_open_restaurants_df['business_id'].isin(user_info['business_id'])]['city'].value_counts().head(5).to_string()}")
print()
test_svd_recommendations(svd, sample_user, yelp_open_restaurants_df, train_df, k=10)

In [ ]:
print(sorted(yelp_open_restaurants_df["city"].unique().tolist()))
print(f"\nTotal cities: {yelp_open_restaurants_df['city'].nunique()}")

In [ ]:
pd.set_option('display.max_rows', None)
print(yelp_open_restaurants_df["city"].value_counts())
pd.reset_option('display.max_rows')


In [ ]:
city_counts = yelp_open_restaurants_df["city"].value_counts()
print(city_counts)
print(f"\nTotal cities: {len(city_counts)}")


In [ ]:
import re

# Normalization map for known aliases
CITY_ALIASES = {
    # Philadelphia variants
    "phila": "Philadelphia", "philly": "Philadelphia", "philadelphia ": "Philadelphia",
    # St. Louis variants
    "st louis": "St. Louis", "saint louis": "St. Louis", "st. louis": "St. Louis",
    "st louis county": "St. Louis", "st louis downtown": "St. Louis",
    # St. Petersburg variants
    "st. petersburg": "St. Petersburg", "saint petersburg": "St. Petersburg",
    "st petersburg": "St. Petersburg", "st petersberg": "St. Petersburg",
    "st petersurg": "St. Petersburg", "saintt petersburg": "St. Petersburg",
    # Nashville variants
    "nashville-davidson metropolitan government (balance)": "Nashville",
    "east nashville": "Nashville", "nashville ": "Nashville",
    # Tampa variants
    "tampa,": "Tampa", "tampa,fl": "Tampa", "tampa florida": "Tampa",
    "south tampa": "Tampa", "southwest tampa": "Tampa", "tampa palms": "Tampa",
    # Indianapolis variants
    "indianapolis ": "Indianapolis", "indianapolis city (balance)": "Indianapolis",
    "inpolis": "Indianapolis", "indianopolis": "Indianapolis",
    # Boise variants
    "boise city": "Boise", "east boise": "Boise",
    # Reno variants
    "reno ": "Reno", "reno nevada": "Reno",
    # Tucson variants
    "tuscon": "Tucson", "south tucson": "Tucson",
    # New Orleans variants
    "new orleans ": "New Orleans",
    # Clearwater variants
    "clearwater ": "Clearwater", "clearwater beach": "Clearwater",
    "clearwater/ countryside": "Clearwater",
    # Edmonton variants
    "edmonton": "Edmonton", "west edmonton": "Edmonton",
    # Metairie variants
    "metairie ": "Metairie", "metarie": "Metairie",
    # Sparks variants
    "sparks ": "Sparks",
    # Wilmington variants
    "wilmington ": "Wilmington",
    # Goodlettsville
    "goodletsville": "Goodlettsville",
    # La Vergne
    "lavergne": "La Vergne",
    # Mt. Juliet
    "mt juliet": "Mt. Juliet", "mt.juliet": "Mt. Juliet", "mount juliet": "Mt. Juliet",
    # King of Prussia
    "king of prussia": "King of Prussia",
    # Land O Lakes
    "land o lakes": "Land O' Lakes", "land o'lakes": "Land O' Lakes", "land o' lakes": "Land O' Lakes",
    # O'Fallon
    "o fallon": "O'Fallon", "o' fallon": "O'Fallon", "o'fallon": "O'Fallon",
    # St. Pete Beach
    "st pete beach": "St. Pete Beach", "saint pete beach": "St. Pete Beach",
    "st. pete beach": "St. Pete Beach",
    # Wesley Chapel
    "wesley chapel ": "Wesley Chapel", "wesley chapel": "Wesley Chapel",
    # Beech Grove
    "beech grove,": "Beech Grove",
    # Lansdale
    "lansdale ": "Lansdale", "landsdale": "Lansdale",
    # Maryville
    "maryville ": "Maryville",
    # Feasterville-Trevose
    "feasterville-trevose": "Feasterville-Trevose", "feasterville trevose": "Feasterville-Trevose",
    "feasterville-trevose": "Feasterville-Trevose",
    # Cherry Hill
    "cherry hil": "Cherry Hill",
    # Royersford
    "royersford ": "Royersford",
    # Telford
    "telford ": "Telford",
    # Deptford variants
    "deptford township": "Deptford", "deptford twp": "Deptford",
}

def normalize_city(city):
    if not isinstance(city, str):
        return "Unknown"
    cleaned = city.strip().lower()
    if cleaned in CITY_ALIASES:
        return CITY_ALIASES[cleaned]
    return city.strip().title()

yelp_open_restaurants_df["city_clean"] = yelp_open_restaurants_df["city"].apply(normalize_city)

# Get cities with 100+ businesses for the dropdown
city_counts_clean = yelp_open_restaurants_df["city_clean"].value_counts()
AVAILABLE_CITIES = sorted(city_counts_clean[city_counts_clean >= 100].index.tolist())

print(f"Cities available for dropdown ({len(AVAILABLE_CITIES)}):")
for city in AVAILABLE_CITIES:
    print(f"  {city}: {city_counts_clean[city]} businesses")


In [ ]:
user_features = pd.read_parquet(f"{origin_dir}/users.parquet")
train_reviews = pd.read_parquet(f"{origin_dir}/train_reviews.parquet")

def get_user_context(user_id):
    user = user_features[user_features["user_id"] == user_id].iloc[0]

    # join train reviews with business cities
    user_reviews = train_reviews[train_reviews["user_id"] == user_id]
    user_reviews_with_city = user_reviews.merge(
        yelp_open_restaurants_df[["business_id", "city_clean"]],
        on="business_id",
        how="left"
    )
    top_cities = user_reviews_with_city["city_clean"].value_counts().head(3).index.tolist()

    return {
        "avg_stars": user["avg_stars_given"],
        "review_count": user["review_count_log"],
        "is_elite": user["is_elite"],
        "top_cities": top_cities,
    }

# test with a real user_id
sample_user = train_reviews["user_id"].iloc[0]
get_user_context(sample_user)